# Digital to Analog Clock Converter

## Project Overview
In this notebook, we will train a Deep Learning model (U-Net) to translate images of **Digital Clocks** into **Analog Clocks**.

### Pipeline Steps:
1. **Setup:** Import libraries and configure the device (GPU/CPU).
2. **Data Inspection:** Visualize our synthetic training data.
3. **Model Initialization:** Load our U-Net architecture.
4. **Training:** Train the model and visualize progress in real-time.
5. **Evaluation:** Test the model on unseen data.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Import our custom modules from src/
from src.dataset import ClockDataset, get_transforms
from src.model import ClockTranslator

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading & Visualization
Let's load our training dataset and visualize a batch to ensure the inputs (Digital) and targets (Analog) are correct.

In [ ]:
# Hyperparameters
BATCH_SIZE = 16
IMG_SIZE = 256

# Load Dataset
train_dataset = ClockDataset(root_dir='data/train', transform=get_transforms(IMG_SIZE))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = ClockDataset(root_dir='data/test', transform=get_transforms(IMG_SIZE))
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print(f"Training Samples: {len(train_dataset)}")
print(f"Testing Samples: {len(test_dataset)}")

In [ ]:
def show_batch(loader):
    dig, ana = next(iter(loader))
    
    fig, axes = plt.subplots(4, 2, figsize=(8, 16))
    for i in range(4):
        # Denormalize: (x * 0.5) + 0.5 to get back to [0, 1]
        d_img = dig[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
        a_img = ana[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
        
        axes[i, 0].imshow(d_img)
        axes[i, 0].set_title("Input: Digital")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(a_img)
        axes[i, 1].set_title("Target: Analog")
        axes[i, 1].axis('off')
    plt.show()

show_batch(train_loader)

## 2. Model Initialization
We initialize our U-Net model and the optimizer.

In [ ]:
model = ClockTranslator().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.L1Loss() # L1 Loss encourages sharp images

# Check model structure (optional)
# print(model)

## 3. Training Loop
We will train the model and visualize the results after every few epochs.

In [ ]:
def visualize_prediction(model, loader):
    model.eval()
    with torch.no_grad():
        dig, ana = next(iter(loader))
        dig = dig.to(device)
        
        fake_ana = model(dig)
        
        # Move to CPU for plotting
        dig = dig.cpu()
        fake_ana = fake_ana.cpu()
        ana = ana.cpu()
        
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        
        # Digital Input
        axes[0].imshow(dig[0].permute(1, 2, 0) * 0.5 + 0.5)
        axes[0].set_title("Input")
        axes[0].axis('off')

        # Generated Analog
        axes[1].imshow(fake_ana[0].permute(1, 2, 0) * 0.5 + 0.5)
        axes[1].set_title("Generated")
        axes[1].axis('off')

        # Real Analog
        axes[2].imshow(ana[0].permute(1, 2, 0) * 0.5 + 0.5)
        axes[2].set_title("Ground Truth")
        axes[2].axis('off')
        
        plt.show()
    model.train()

In [ ]:
NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    loop = tqdm(train_loader, leave=True)
    
    for idx, (dig, ana) in enumerate(loop):
        dig = dig.to(device)
        ana = ana.to(device)
        
        # Forward
        outputs = model(dig)
        loss = criterion(outputs, ana)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Update progress bar
        loop.set_description(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
        loop.set_postfix(loss=loss.item())
        
    # Visualize progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Example Output at Epoch {epoch+1}:")
        visualize_prediction(model, test_loader)
        
    # Save model checkpoint
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f"models/unet_epoch_{epoch+1}.pth")

## 4. Final Evaluation
Now that training is done, let's look at more examples from the test set.

In [ ]:
visualize_prediction(model, test_loader)